# SC-22-Solana-Anchor-Rust - Solana avec Anchor

[<< Move & Sui](SC-21-Move-Sui-Python.ipynb) | [Cross-Chain >>](../06-Real-World/SC-23-Cross-Chain-Python.ipynb) | [jumeau Python](SC-22-Solana-Anchor-Python.ipynb)

***

## Objectifs d'apprentissage

1. Comprendre l'architecture **Solana** (accounts, programs)
2. Utiliser le framework **Anchor** (Rust)
3. Créer des **PDAs** (Program Derived Addresses)
4. Implementer des **CPIs** (Cross-Program Invocations)

### Ce que Rust apporte

Un programme Anchor **s'écrit en Rust** : ce carnet emploie le langage du programme on-chain lui-même, là où le jumeau Python le simule. Les modèles exécutés ici sont écrits avec les types d'un programme sous `#[program]` :
- la dérivation d'un PDA, par un SHA-256 écrit en `std` (exemple guidé 1) ;
- les tailles de compte, en `u64` (exercice de la section 4) ;
- le Counter de l'exemple guidé 3, avec `struct`, `u64` et `Result`.

Les programmes Anchor complets restent affichés comme texte. Le carnet ne compile pas `anchor_lang`, qui demande la chaîne d'outils Solana (voir section 5).

### Prerequis

- [SC-3](../01-Solidity-Foundation/SC-03-Solidity-Basics-Python.ipynb) a [SC-6](../01-Solidity-Foundation/SC-06-Errors-Events-Python.ipynb) completes (concepts)
- Notions de base en Rust (variables, ownership, structs)
- Solana CLI et Anchor installe (voir setup dans le notebook)

### Duree estimee : 55 minutes

***

## 1. Architecture Solana

Solana utilise un modèle différent d'Ethereum.

In [2]:
// Comparaison Ethereum vs Solana
println!("{}", r#"
ETHEREUM vs SOLANA

| Concept       | Ethereum              | Solana                  |
|--------------|----------------------|-------------------------|
| Execution    | EVM                   | BPF (Berkeley Packet Filter) |
| State        | Storage in contract  | Accounts (separates)    |
| Langage      | Solidity              | Rust (via Anchor)       |
| Adressage    | Nonces sequentielles  | Hash d'accounts         |
| Fees         | Gas (variable)        | Compute units + rent    |
| TPS          | ~15-30                | ~65,000                 |

COMPOSANTS SOLANA:
- Program    : Code executable (stateless)
- Account    : Stockage de donnees (state)
- PDA        : Program Derived Address
- CPI        : Cross-Program Invocation
- Token      : SPL Token (equivalent ERC-20/721)
"#);

ETHEREUM vs SOLANA


| Concept       | Ethereum              | Solana                  |


|--------------|----------------------|-------------------------|


| Execution    | EVM                   | BPF (Berkeley Packet Filter) |


| State        | Storage in contract  | Accounts (separates)    |


| Langage      | Solidity              | Rust (via Anchor)       |


... (5 outputs left out)

- Account    : Stockage de donnees (state)


- PDA        : Program Derived Address


- CPI        : Cross-Program Invocation


- Token      : SPL Token (equivalent ERC-20/721)


***

## 2. Programme Anchor Simple

In [3]:
// Counter en Anchor
const ANCHOR_COUNTER: &str = r#"
// Fichier: programs/counter/src/lib.rs

use anchor_lang::prelude::*;

declare_id!("Counter111111111111111111111111111111111111");

#[program]
pub mod counter {
    use super::*;

    pub fn initialize(ctx: Context<Initialize>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count = 0;
        counter.authority = ctx.accounts.authority.key();
        Ok(())
    }

    pub fn increment(ctx: Context<Increment>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count += 1;
        Ok(())
    }

    pub fn decrement(ctx: Context<Decrement>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        require!(counter.count > 0, CounterError::Underflow);
        counter.count -= 1;
        Ok(())
    }
}

#[derive(Accounts)]
pub struct Initialize<'info> {
    #[account(
        init,
        payer = authority,
        space = 8 + Counter::INIT_SPACE,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    #[account(mut)]
    pub authority: Signer<'info>,
    pub system_program: Program<'info, System>,
}

#[derive(Accounts)]
pub struct Increment<'info> {
    #[account(
        mut,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    pub authority: Signer<'info>,
}

#[derive(Accounts)]
pub struct Decrement<'info> {
    #[account(
        mut,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    pub authority: Signer<'info>,
}

#[account]
#[derive(InitSpace)]
pub struct Counter {
    pub authority: Pubkey,
    pub count: u64,
}

#[error_code]
pub enum CounterError {
    #[msg("Counter cannot go below zero")]
    Underflow,
}
"#;
println!("Programme Counter Anchor:");
println!("{}", ANCHOR_COUNTER);

Programme Counter Anchor:


// Fichier: programs/counter/src/lib.rs


use anchor_lang::prelude::*;


declare_id!("Counter111111111111111111111111111111111111");


#[program]


pub mod counter {


... (60 outputs left out)

pub enum CounterError {


    #[msg("Counter cannot go below zero")]


    Underflow,


}


### Interpretation : Architecture Solana vs Ethereum

Ce tableau compare les différents modèles d'exécution et de stockage entre Ethereum et Solana.

| Concept | Ethereum | Solana | Impact pratique |
|---------|----------|---------|-----------------|
| **Exécution** | EVM (bytecode stack-based) | BPF (Berkeley Packet Filter, compile depuis Rust/C++) | Solana est plus rapide (JIT, native compilation) |
| **State** | Stocke dans le contrat (storage) | Comptes separes du programme | Solana : plus de parallelisme, état unique répliqué (pas de sharding) |
| **Langage** | Solidity (type, haut niveau) | Rust (système, verifie) | Rust est plus strict (ownership) mais plus performant |
| **Adressage** | Nonces séquentielles (contract creation count) | Hash des seeds du PDA | Solana : addresses déterministes, previsibles |
| **Fees** | Gas (variable selon complexite) | Compute units + rent exemption | Solana : couts predictibles, rent pour stockage durable |
| **TPS** | ~15-30 (Layer 1) | ~65,000 (théorique) | Solana : haut debit, pas de sharding (état unique répliqué) |

**Points clés** :
1. **Programs stateless** : Le code est separe des données (comptes). Le programme ne contient pas d'état.
2. **Accounts model** : Chaque compte est une entite separee avec son propre solde (Lamports) et ses données.
3. **Rent exemption** : Les comptes doivent avoir un solde minimum pour exister (rent). Au-dessus de ce seuil, pas de frais de stockage.
4. **Parallelisme** : Solana peut executer plusieurs transactions en parallele si elles touchent des comptes différents.

**Implications pour les developpeurs** :
- Sur Ethereum : on deploye un contrat avec tout l'état (mappings, arrays)
- Sur Solana : on deploye un programme (code) et on créé des comptes PDAs pour chaque utilisateur/entite
- Le modèle Solana permet plus de parallelisme mais demande plus de planification (structure des comptes)

> **Note technique** : BPF (Berkeley Packet Filter) est un format bytecode securise. Le runtime qui execute les programmes Solana, **sBPF** (`anza-xyz/sbpf`), est une **machine virtuelle en espace utilisateur** : un interpreteur et un compilateur JIT x86_64. Aucun bytecode de programme Solana ne s'execute dans le noyau Linux.

***

## 3. PDAs (Program Derived Addresses)

Les PDAs sont des adresses derivees déterministes.

In [4]:
// PDA explanation
println!("{}", r#"
PDA (Program Derived Address)

PROPRIETES:
- Adresse derivee du program_id et de seeds
- Pas de cle privee (programme seulement)
- Deterministe: meme seeds = meme PDA
- Bump: [b"seed", bump] pour trouver le PDA

GENERATION:
seeds = [b"counter", authority.key().as_ref()]
(pda, bump) = Pubkey::find_program_address(&seeds, program_id)

UTILISATION:
1. Stockage de donnees utilisateur
2. Escrows
3. Token accounts
4. Metadata

EXEMPLE DANS ANCHOR:
#[account(
    seeds = [b"counter", authority.key().as_ref()],
    bump
)]
pub counter: Account<'info, Counter>,
"#);

PDA (Program Derived Address)


PROPRIETES:


- Adresse derivee du program_id et de seeds


- Pas de cle privee (programme seulement)


- Deterministe: meme seeds = meme PDA


- Bump: [b"seed", bump] pour trouver le PDA


... (10 outputs left out)

    seeds = [b"counter", authority.key().as_ref()],


    bump


)]


pub counter: Account<'info, Counter>,


### Interpretation : Programme Counter Anchor

Ce code presente un programme de compteur simple avec les concepts fondamentaux d'Anchor.

| Composant | Rôle | Details |
|-----------|------|---------|
| `#[program]` | Module de fonctions publiques | Points d'entree du programme (appelables par les clients) |
| `Context<Initialize>` | Validation des comptes | Anchor verifie que tous les comptes passes sont valides |
| `#[account(init)]` | Création de compte | Alloue un nouveau compte PDA avec `init` + `payer` + `space` + `seeds` |
| `#[account(mut)]` | Compte modifiable | Le compte peut etre modifie pendant la transaction |
| `#[error_code]` | Erreurs custom | Erreurs spécifiques au programme (ici: `Underflow`) |

**Points clés** :
1. `initialize` créé un PDA `counter` avec seeds `[b"counter", authority.key().as_ref()]` → 1 counter par utilisateur
2. `space = 8 + Counter::INIT_SPACE` : 8 octets de discriminant Anchor + taille de la struct `Counter` (32 + 8 = 40)
3. `increment` et `decrement` utilisent les mêmes seeds pour retrouver le PDA du counter
4. `require!(counter.count > 0, CounterError::Underflow)` : protection contre underflow (custom error)

**Comparaison Solidity vs Anchor** :
- **Solidity** : `mapping(address => uint256) public counters;` (stocke dans le contrat)
- **Anchor** : Chaque utilisateur a son propre compte PDA `Counter` (separe du programme)

**Avantages du modèle Solana** :
- Parallelelisme : chaque utilisateur modifie son propre compte (pas de contention)
- Rent exemption : l'utilisateur paie pour son propre compte (pas le deployeur)
- Flexibilite : les comptes peuvent etre fermes pour recuperer la rent

> **Note technique** : Le discriminant Anchor (8 octets) est un hash unique qui identifie le type de compte. Il permet a Anchor de verifier qu'un compte passe est bien du type attendu.

### Exercice : Générateur de struct Anchor

Ecrire un programme Anchor pour gerer un profil utilisateur sur Solana. Completez la fonction `generate_anchor_profile` qui produit le code Rust du programme.

**Objectifs** :
1. Définir une struct `UserProfile` avec les champs : owner (Pubkey), username (String[32]), bio (String[280]), reputation (u64)
2. Implementer les fonctions `create_profile` et `update_reputation`
3. Calculer l'espace necessaire pour le compte

**Indices** :
- Utiliser `#[account(init, payer = owner, space = 8 + UserProfile::INIT_SPACE, seeds = [...], bump)]` pour la création
- Le PDA est derive de `b"profile"` et de l'adresse du proprietaire
- `update_reputation` prend un montant positif ou negatif (utiliser `i64` pour l'argument)

In [5]:
// Generer le code Rust d'un programme Anchor UserProfile.
//
// Le programme doit definir :
// - Struct UserProfile : owner (Pubkey), username (String[32]), bio (String[280]), reputation (u64)
// - Fonction create_profile : initialise le profil avec username et bio
// - Fonction update_reputation : ajoute un delta (positif ou negatif) a la reputation
//
// Retourner le code Rust sous forme de chaine.
//
// TODO etudiant : implementez generate_anchor_profile.
// Indice : inspirez-vous du programme Counter vu en section 2.
// Les seeds du PDA seront [b"profile", owner.key().as_ref()].
fn generate_anchor_profile() -> String {
    // TODO etudiant : ecrivez le code Rust complet
    String::new() // TODO etudiant : remplacez par le code du programme
}

// Test : afficher le code genere
let code = generate_anchor_profile();
if !code.is_empty() {
    println!("Programme UserProfile Anchor :");
    println!("{}", code);
} else {
    println!("Exercice a completer : generate_anchor_profile");
};

Exercice a completer : generate_anchor_profile


***

## 4. CPI (Cross-Program Invocation)

In [6]:
// CPI example
const CPI_EXAMPLE: &str = r#"
// CPI vers System Program pour transfer SOL
use anchor_lang::system_program;

pub fn transfer_sol(ctx: Context<TransferSol>, amount: u64) -> Result<()> {
    let cpi_context = CpiContext::new(
        ctx.accounts.system_program.to_account_info(),
        system_program::Transfer {
            from: ctx.accounts.from.to_account_info(),
            to: ctx.accounts.to.to_account_info(),
        }
    );
    system_program::transfer(cpi_context, amount)
}

#[derive(Accounts)]
pub struct TransferSol<'info> {
    #[account(mut)]
    pub from: Signer<'info>,
    #[account(mut)]
    pub to: AccountInfo<'info>,
    pub system_program: Program<'info, System>,
}

// CPI vers Token Program
use anchor_spl::token::{self, Transfer as TokenTransfer};

pub fn transfer_token(
    ctx: Context<TransferToken>,
    amount: u64
) -> Result<()> {
    let cpi_accounts = TokenTransfer {
        from: ctx.accounts.from.to_account_info(),
        to: ctx.accounts.to.to_account_info(),
        authority: ctx.accounts.authority.to_account_info(),
    };
    let cpi_program = ctx.accounts.token_program.to_account_info();
    token::transfer(CpiContext::new(cpi_program, cpi_accounts), amount)
}
"#;
println!("CPI Examples:");
println!("{}", CPI_EXAMPLE);

CPI Examples:


// CPI vers System Program pour transfer SOL


use anchor_lang::system_program;


pub fn transfer_sol(ctx: Context<TransferSol>, amount: u64) -> Result<()> {


    let cpi_context = CpiContext::new(


        ctx.accounts.system_program.to_account_info(),


... (25 outputs left out)

    };


    let cpi_program = ctx.accounts.token_program.to_account_info();


    token::transfer(CpiContext::new(cpi_program, cpi_accounts), amount)


}


### Interpretation : PDAs (Program Derived Addresses)

Les PDAs sont une innovation unique de Solana : des adresses sans cle privee, controlees par des programmes.

| Propriete | Description | Impact |
|-----------|-------------|--------|
| **Pas de cle privee** | Le PDA est derive de `program_id + seeds` | Seul le programme peut signer pour le PDA |
| **Déterministe** | Mêmes seeds = même PDA (toujours) | Pas de collision, pas de generation aléatoire |
| **Bump canonique** | Trouve le premier `bump` (1-255) qui donne une adresse hors courbe ed25519 | Garantit que seul le programme peut signer |

**Points clés** :
1. `seeds = [b"counter", authority.key().as_ref()]` : le PDA est unique par utilisateur
2. Le `bump` est la valeur qui fait que l'adresse est "hors courbe" (pas de cle privee correspondante)
3. Anchor calcule automatiquement le bump avec `#[account(seeds = [...], bump)]`
4. Les PDAs sont utilises pour : comptes de données utilisateur, vaults, escrows, metadata NFT

**Cas d'usage typiques** :
- **Compte utilisateur** : `seeds = [b"user", user_pubkey.as_ref()]` → 1 compte par utilisateur
- **Vault d'escrow** : `seeds = [b"vault", escrow_id.as_ref()]` → vault unique par escrow
- **Metadata NFT** : `seeds = [b"metadata", mint.as_ref()]` → metadata unique par mint

**Generation manuelle** (Rust) :
```rust
let (pda, bump) = Pubkey::find_program_address(
    &[b"counter", authority.key().as_ref()],
    program_id
);
```

> **Note technique** : Le "bump" est entre 1 et 255. Si 255 ne suffit pas, il n'y a pas de PDA valide. Anchor utilise le "bump canonique" (le premier qui marche) pour garantir l'unicite.

### Exercice : Calculateur de PDA et espace de compte

Sur Solana, chaque compte doit allouer un espace de stockage payant (rent). La taille depend des champs de la struct Anchor. Completez les fonctions `calculate_pda` et `calculate_account_size`.

**Objectifs** :
1. Deriver une PDA a partir de seeds et d'un program_id (simulation)
2. Calculer l'espace necessaire pour un compte Anchor (discriminant 8 octets + taille des champs)
3. Estimer le cout en SOL (rent) pour créer le compte

**Indices** :
- L'espace Anchor = 8 octets (discriminant) + somme des tailles des champs
- Types Anchor : Pubkey = 32 octets, u64 = 8 octets, bool = 1 octet, String = 4 + longueur, Vec<T> = 4 + N * taille(T)
- Le rent sur Solana est de 6960 lamports par octet de données (0.000006960 SOL/octet), et le compte paie en plus 128 octets d'overhead de stockage

In [7]:
// Tailles des types Anchor (en octets)
const ANCHOR_TYPE_SIZES: [(&str, u64); 9] = [
    ("Pubkey", 32), ("u64", 8), ("u32", 4), ("u16", 2), ("u8", 1),
    ("i64", 8), ("i32", 4), ("bool", 1), ("f64", 8),
];

// Taux reel de rent exemption : 6960 lamports par octet de donnees.
// Source : anza-xyz/solana-sdk, rent/src/lib.rs --
//   DEFAULT_LAMPORTS_PER_BYTE = 6_960 (l. 58)
//   ACCOUNT_STORAGE_OVERHEAD  = 128 (l. 86)
// Le cout total d'un compte est (128 + taille_donnees) * 6960 lamports :
// les 128 octets d'overhead s'ajoutent au volume de donnees, ils ne le
// remplacent pas.
const LAMPORTS_PER_BYTE: u64 = 6_960;
const ACCOUNT_STORAGE_OVERHEAD: u64 = 128;
const RENT_PER_BYTE: f64 = 6960.0 / 1e9; // SOL par octet de donnees

fn calculate_pda(seeds: &[&[u8]], program_id: &str) -> (String, u8) {
    // TODO etudiant : implementez calculate_pda
    (String::new(), 0) // TODO etudiant
}

fn calculate_account_size(fields: &[(&str, &str, Option<u64>)]) -> u64 {
    // TODO etudiant : implementez calculate_account_size
    0 // TODO etudiant
}

// Test : compte Counter avec authority + count
let fields: [(&str, &str, Option<u64>); 2] = [
    ("authority", "Pubkey", None),
    ("count", "u64", None),
];
let size = calculate_account_size(&fields);
let (pda, bump) = calculate_pda(&[b"counter", b"alice_pubkey"], "Counter111111111111111111111111111111111111");
let rent: f64 = if size > 0 { (ACCOUNT_STORAGE_OVERHEAD + size) as f64 * RENT_PER_BYTE } else { 0.0 };
println!("Espace compte : {} octets", size);
println!("PDA simulee : {}...", pda.chars().take(32).collect::<String>());
println!("Bump retenu : {}", bump);
println!("Rent estime : {:.6} SOL", rent);
println!("Exercice a completer : calculate_pda et calculate_account_size");

Espace compte : 0 octets


PDA simulee : ...


Bump retenu : 0


Rent estime : 0.000000 SOL


Exercice a completer : calculate_pda et calculate_account_size


***

## 5. Commandes Anchor CLI

In [8]:
// Anchor CLI commands
println!("{}", r#"
INSTALLATION:
cargo install --git https://github.com/coral-xyz/anchor avm --locked --force
avm install latest
avm use latest

CREATION PROJET:
anchor init my_program
cd my_program

STRUCTURE:
my_program/
|-- Anchor.toml          # Configuration
|-- Cargo.toml
|-- programs/
|   `-- my_program/
|       |-- Cargo.toml
|       `-- src/
|           `-- lib.rs   # Code principal
|-- tests/
|   `-- my_program.ts    # Tests
`-- app/                 # Frontend

COMMANDES:
anchor build              # Compiler
anchor test               # Executer les tests
anchor deploy             # Deploier
anchor run <script>       # Executer un script

CLUSTER:
anchor test --skip-local-validator  # Avec validator externe
anchor deploy --provider.cluster devnet
"#);

INSTALLATION:


cargo install --git https://github.com/coral-xyz/anchor avm --locked --force


avm install latest


avm use latest


CREATION PROJET:


anchor init my_program


... (17 outputs left out)

anchor run <script>       # Executer un script


CLUSTER:


anchor test --skip-local-validator  # Avec validator externe


anchor deploy --provider.cluster devnet


### Interpretation : Cross-Program Invocations (CPI)

Les CPIs permettent a un programme Solana d'appeler un autre programme (équivalent des `delegatecall` en Solidity).

| Type de CPI | Programme cible | Opération |
|-------------|----------------|-----------|
| `system_program::transfer` | System Program | Transferts de SOL native |
| `token::transfer` | SPL Token Program | Transferts de tokens SPL |
| Metaplex CPI | Metaplex | Création NFT, metadata |
| Custom CPI | Autres programmes utilisateur | Composition de fonctionnalites |

**Points clés** :
1. `CpiContext::new` créé un contexte CPI standard (signataire = l'appelant)
2. `CpiContext::new_with_signer` permet a un PDA de signer (cas du vault escrow)
3. Tous les comptes necessaires doivent etre passes dans la struct `Accounts` (Anchor verifie la coherence)
4. Les CPIs sont payables par l'appelant (compute units + rent exemption)

**Différence EVM vs Solana** :
- EVM : `delegatecall` execute le code d'un autre contrat DANS le contexte du contrat appelant
- Solana : CPI execute le code d'un autre programme AVEC ses propres comptes (contexte separe)

**Pattern CPI classique** :
1. Définir les comptes necessaires dans la struct `#[derive(Accounts)]`
2. Créer le `CpiContext` avec les bons comptes et autorites
3. Appeler la fonction du programme cible (ex: `token::transfer`)

> **Note technique** : Les CPIs sont COMPOSES sur Solana. Un programme A peut appeler B, qui appelle C, etc. La limite de profondeur est de 4 appels chaînes.

### Exercice : Analyseur de structure de comptes Anchor

L'avantage principal de Solana sur Ethereum est le parallelisme des transactions : deux transactions qui modifient des comptes différents peuvent s'executer en parallele. Completez la fonction `analyze_parallelism` qui determine quelles transactions peuvent s'executer simultanement.

**Objectifs** :
1. Identifier les comptes en lecture et ecriture pour chaque instruction
2. Determiner quelles paires de transactions sont parallelisables
3. Calculer le throughput théorique (nombre de transactions executables en parallele)

**Indices** :
- Deux transactions sont parallelisables si elles ne partagent **aucun** compte, quel que soit son sens d'acces : un compte lu par l'une et ecrit par l'autre les serialise aussi
- Un compte lu par les deux transactions ne bloque pas le parallelisme
- Le throughput théorique = nombre maximum de transactions parallelisables dans un même batch

In [9]:
struct Tx {
    name: &'static str,
    reads: Vec<&'static str>,
    writes: Vec<&'static str>,
}

struct Analyse {
    parallelizable_pairs: Vec<(usize, usize)>,
    conflicts: Vec<(usize, usize)>,
    max_parallel: usize,
    sequential_only: Option<bool>,
}

fn py_opt_bool(b: Option<bool>) -> String {
    match b {
        None => "None".to_string(),
        Some(true) => "True".to_string(),
        Some(false) => "False".to_string(),
    }
}

// Analyser le parallelisme possible entre transactions Solana.
//
// Args:
//     transactions: liste de Tx, chaque Tx contient :
//         - name: nom de la transaction
//         - reads: comptes lus
//         - writes: comptes ecrits
//         Un compte present dans les deux est un acces en ecriture.
//
// Returns:
//     Analyse avec :
//         - parallelizable_pairs: liste de tuples (i, j) parallelisables
//         - conflicts: liste de tuples (i, j) en conflit
//         - max_parallel: nombre max de tx parallelisables
//         - sequential_only: True si aucune paire n'est parallelisable
//
// TODO etudiant : implementez analyze_parallelism.
// Etape 1 : pour chaque paire (i, j), verifier si les comptes cites par les deux
//           transactions s'intersectent (reads ET writes confondus)
// Etape 2 : si l'intersection est vide, la paire est parallelisable
//           (le runtime reserve un compte en ecriture ET un compte en lecture :
//           un acces en ecriture exclut tout acces concurrent, meme en lecture)
// Etape 3 : calculer max_parallel (taille du plus grand sous-ensemble mutuellement parallelisable)
fn analyze_parallelism(transactions: &[Tx]) -> Analyse {
    // TODO etudiant : implementez analyze_parallelism
    println!("Exercice a completer : analyze_parallelism");
    Analyse {
        parallelizable_pairs: vec![],
        conflicts: vec![],
        max_parallel: 0,
        sequential_only: None,
    }
}

// Exemple : transactions Solana typiques
let txs: [Tx; 5] = [
    Tx { name: "increment_counter_Alice", reads: vec!["counter_alice"], writes: vec!["counter_alice"] },
    Tx { name: "increment_counter_Bob", reads: vec!["counter_bob"], writes: vec!["counter_bob"] },
    Tx { name: "transfer_SOL_Alice_to_Bob", reads: vec!["alice_wallet", "bob_wallet"], writes: vec!["alice_wallet", "bob_wallet"] },
    Tx { name: "mint_token", reads: vec!["mint_authority"], writes: vec!["mint_authority", "token_account"] },
    Tx { name: "read_counter_Alice", reads: vec!["counter_alice"], writes: vec![] },
];

let result = analyze_parallelism(&txs);
println!("Paires parallelisables : {}", result.parallelizable_pairs.len());
println!("Conflits : {}", result.conflicts.len());
println!("Max parallel : {}", result.max_parallel);
println!("Sequential only : {}", py_opt_bool(result.sequential_only));

Exercice a completer : analyze_parallelism


Paires parallelisables : 0


Conflits : 0


Max parallel : 0


Sequential only : None


***

## 6. Exemples guidés

In [10]:
// Exercice: Token Escrow
const EXERCISE_ESCROW: &str = r#"
use anchor_lang::prelude::*;
use anchor_spl::token::{self, Mint, Token, TokenAccount};

#[program]
pub mod escrow {
    use super::*;

    pub fn make(
        ctx: Context<Make>,
        seed: u64,
        receive_amount: u64,
    ) -> Result<()> {
        let escrow = &mut ctx.accounts.escrow;
        escrow.seed = seed;
        escrow.maker = ctx.accounts.maker.key();
        escrow.mint_a = ctx.accounts.mint_a.key();
        escrow.mint_b = ctx.accounts.mint_b.key();
        escrow.receive_amount = receive_amount;
        escrow.bump = ctx.bumps.escrow;

        // Transfer tokens to vault
        token::transfer(
            CpiContext::new(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.deposit.to_account_info(),
                    to: ctx.accounts.vault.to_account_info(),
                    authority: ctx.accounts.maker.to_account_info(),
                },
            ),
            ctx.accounts.deposit.amount,
        )
    }

    pub fn take(ctx: Context<Take>) -> Result<()> {
        // Transfer B to maker
        token::transfer(
            CpiContext::new(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.depositor_token_b.to_account_info(),
                    to: ctx.accounts.maker_token_b.to_account_info(),
                    authority: ctx.accounts.depositor.to_account_info(),
                },
            ),
            ctx.accounts.escrow.receive_amount,
        )?;

        // Transfer A to depositor
        token::transfer(
            CpiContext::new_with_signer(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.vault.to_account_info(),
                    to: ctx.accounts.depositor_token_a.to_account_info(),
                    authority: ctx.accounts.escrow.to_account_info(),
                },
                &[&[
                    b"escrow",
                    ctx.accounts.maker.key().as_ref(),
                    &ctx.accounts.escrow.seed.to_le_bytes()[..],
                    &[ctx.accounts.escrow.bump],
                ]],
            ),
            ctx.accounts.vault.amount,
        )
    }
}

// --- Structs d'accounts (absentes de la version initiale) ---
// Sans ces structs, `Context<Make>` et `Context<Take>` ne compilent pas :
// Anchor resout ces types vers des structs `#[derive(Accounts)]` du meme nom.
#[derive(Accounts)]
pub struct Make<'info> {
    #[account(
        init,
        payer = maker,
        space = 8 + Escrow::INIT_SPACE,
        seeds = [b"escrow", maker.key().as_ref(), &seed.to_le_bytes()],
        bump
    )]
    pub escrow: Account<'info, Escrow>,
    #[account(mut)]
    pub maker: Signer<'info>,
    pub mint_a: Account<'info, Mint>,
    pub mint_b: Account<'info, Mint>,
    #[account(mut)]
    pub deposit: Account<'info, TokenAccount>,
    #[account(mut)]
    pub vault: Account<'info, TokenAccount>,
    pub token_program: Program<'info, Token>,
    pub system_program: Program<'info, System>,
}

#[derive(Accounts)]
pub struct Take<'info> {
    #[account(
        mut,
        seeds = [b"escrow", maker.key().as_ref(), &escrow.seed.to_le_bytes()],
        bump = escrow.bump
    )]
    pub escrow: Account<'info, Escrow>,
    /// CHECK: compte de l'autorite du compte escrow, verifie par les seeds.
    pub maker: UncheckedAccount<'info>,
    #[account(mut)]
    pub depositor: Signer<'info>,
    #[account(mut)]
    pub depositor_token_a: Account<'info, TokenAccount>,
    #[account(mut)]
    pub depositor_token_b: Account<'info, TokenAccount>,
    #[account(mut)]
    pub maker_token_b: Account<'info, TokenAccount>,
    #[account(mut)]
    pub vault: Account<'info, TokenAccount>,
    pub token_program: Program<'info, Token>,
}

#[account]
#[derive(InitSpace)]
pub struct Escrow {
    pub maker: Pubkey,
    pub mint_a: Pubkey,
    pub mint_b: Pubkey,
    pub receive_amount: u64,
    pub seed: u64,
    pub bump: u8,
}
"#;
println!("Exercice Escrow:");
println!("{}", EXERCISE_ESCROW);

Exercice Escrow:


use anchor_lang::prelude::*;


use anchor_spl::token::{self, Mint, Token, TokenAccount};


#[program]


pub mod escrow {


    use super::*;


... (110 outputs left out)

    pub receive_amount: u64,


    pub seed: u64,


    pub bump: u8,


}


### Interpretation : Commandes Anchor CLI

Ces commandes couvrent le cycle de vie complet de développement d'un programme Solana avec Anchor.

| Commande | Action | Usage typique |
|----------|--------|---------------|
| `anchor init my_program` | Créer un nouveau projet | Demarrage d'un nouveau programme |
| `anchor build` | Compiler le programme Rust | Après modification du code |
| `anchor test` | Executer les tests TypeScript | Verification avant deploiement |
| `anchor deploy` | Deploier sur le cluster configure | Mise en production (devnet/mainnet) |
| `anchor run <script>` | Executer un script TypeScript | Opérations one-off (mint, init, etc.) |

**Points clés** :
1. La structure de projet separe le code (`programs/`) des tests (`tests/`) et du frontend (`app/`)
2. `Anchor.toml` contient la configuration du cluster (localnet, devnet, mainnet) et les features du programme
3. Les tests sont ecrits en TypeScript (avec `@coral-xyz/anchor`) et interagissent avec le programme comme un client le ferait
4. `--skip-local-validator` permet de se connecter a un validator externe (devnet Solana publique) au lieu de lancer un validator local

**Workflow typique** :
```
anchor init mon_program    # Créer le projet
anchor build                # Compiler (verifier la syntaxe Rust)
anchor test                # Executer les tests (verifier la logique)
anchor deploy               # Deploier sur devnet
anchor run scripts/mint.ts  # Executer des opérations d'initialisation
```

> **Note technique** : Anchor utilise `avm` (Anchor Version Manager) pour gerer les versions multiples. `avm install latest` installe la dernière version, `avm use latest` l'active.

***

## 7. Contributions etudiantes -- exemples guides

Les trois exemples ci-dessous, contribues par un groupe d'etudiants, illustrent les concepts fondamentaux de Solana et Anchor vus dans ce notebook : derivation de PDA, structure d'un programme Anchor, et modèle de comptes Solana. Chaque enonce est suivi de sa solution commentee.

### Exemple guide 1 : Simulation de la derivation d'un PDA

_Exemple contribue par le groupe **Mehdi Azouz, Ryad Gazenay & Hani Bouzida** (PR #2189)._

Un PDA (Program Derived Address) est obtenu en cherchant un *bump* (255 -> 0) tel que le 
hash des seeds soit "hors courbe" ed25519 -- ici simule par la condition `premier octet < 128`. 
La derivation doit etre **déterministe** (mêmes seeds -> même PDA) et **unique par utilisateur** 
(seeds différentes -> PDAs différents).

In [11]:
// Exercice 1 : Simulation de derivation PDA

// SHA-256 (FIPS 180-4) en Rust std seul -- bloc a recopier tel quel dans chaque cellule qui hache.
// Verifie contre hashlib.sha256 (CPython) par le script de rejeu de la page.
fn sha256(data: &[u8]) -> [u8; 32] {
    const K: [u32; 64] = [
        0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
        0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
        0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
        0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
        0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
        0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
        0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
        0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
    ];
    let mut h: [u32; 8] = [0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a, 0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19];
    let mut msg = data.to_vec();
    let bit_len = (data.len() as u64).wrapping_mul(8);
    msg.push(0x80);
    while msg.len() % 64 != 56 {
        msg.push(0);
    }
    msg.extend_from_slice(&bit_len.to_be_bytes());
    for chunk in msg.chunks(64) {
        let mut w = [0u32; 64];
        for i in 0..16 {
            w[i] = u32::from_be_bytes([chunk[4 * i], chunk[4 * i + 1], chunk[4 * i + 2], chunk[4 * i + 3]]);
        }
        for i in 16..64 {
            let s0 = w[i - 15].rotate_right(7) ^ w[i - 15].rotate_right(18) ^ (w[i - 15] >> 3);
            let s1 = w[i - 2].rotate_right(17) ^ w[i - 2].rotate_right(19) ^ (w[i - 2] >> 10);
            w[i] = w[i - 16].wrapping_add(s0).wrapping_add(w[i - 7]).wrapping_add(s1);
        }
        let mut v = h;
        for i in 0..64 {
            let s1 = v[4].rotate_right(6) ^ v[4].rotate_right(11) ^ v[4].rotate_right(25);
            let ch = (v[4] & v[5]) ^ (!v[4] & v[6]);
            let t1 = v[7].wrapping_add(s1).wrapping_add(ch).wrapping_add(K[i]).wrapping_add(w[i]);
            let s0 = v[0].rotate_right(2) ^ v[0].rotate_right(13) ^ v[0].rotate_right(22);
            let maj = (v[0] & v[1]) ^ (v[0] & v[2]) ^ (v[1] & v[2]);
            let t2 = s0.wrapping_add(maj);
            v = [t1.wrapping_add(t2), v[0], v[1], v[2], v[3].wrapping_add(t1), v[4], v[5], v[6]];
        }
        for i in 0..8 {
            h[i] = h[i].wrapping_add(v[i]);
        }
    }
    let mut out = [0u8; 32];
    for i in 0..8 {
        out[4 * i..4 * i + 4].copy_from_slice(&h[i].to_be_bytes());
    }
    out
}

// Equivalent de hashlib.sha256(data).hexdigest() : 64 chiffres hexadecimaux minuscules.
fn sha256_hex(data: &[u8]) -> String {
    sha256(data).iter().map(|b| format!("{:02x}", b)).collect()
}

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

fn derive_pda_simulee(seeds: &[&[u8]], program_id: &str) -> Option<(String, u8)> {
    let seeds_bytes: Vec<u8> = seeds.concat();
    for bump in (0..=255u8).rev() {
        let mut data = seeds_bytes.clone();
        data.extend_from_slice(program_id.as_bytes());
        data.push(bump);
        if sha256(&data)[0] < 128 { return Some((sha256_hex(&data), bump)); }
    }
    None
}

// --- Tests ---
let seeds_alice: [&[u8]; 2] = [b"counter", b"alice_pubkey_example"];
let program_id = "MyCounter1111111111111111111111111111111111";

let (pda, bump) = derive_pda_simulee(&seeds_alice, program_id).unwrap();
println!("PDA alice        : {}...", &pda[..16]);
println!("Bump             : {}", bump);
let (pda2, _) = derive_pda_simulee(&seeds_alice, program_id).unwrap();
println!("Deterministe     : {}", py_bool(pda == pda2));
let seeds_bob: [&[u8]; 2] = [b"counter", b"bob_pubkey_example"];
let (pda_bob, _) = derive_pda_simulee(&seeds_bob, program_id).unwrap();
println!("PDA bob != alice : {}", py_bool(pda != pda_bob));

PDA alice        : 45e0b537dd34d532...


Bump             : 255


Deterministe     : True


PDA bob != alice : True


### Exemple guide 2 : Programme Anchor de vote complet

_Exemple contribue par le groupe **Mehdi Azouz, Ryad Gazenay & Hani Bouzida** (PR #2189)._

Un programme Anchor de sondage : `create_poll` initialise le compte `Poll` (un PDA derive de 
`[b"poll", authority]`), et `vote` incremente le compteur du choix avec garde sur la fermeture 
du sondage et la validite du choix. La fermeture est une instruction a part entiere, 
`close_poll`, reservee a l'autorite du sondage : sans elle, la garde `PollClosed` ne 
pourrait jamais mordre. Le code Rust complet est presente ci-dessous.

In [12]:
// Exercice 2 : Programme Anchor de Vote complete (Rust)

const VOTE_PROGRAM_COMPLET: &str = r#"
use anchor_lang::prelude::*;

declare_id!("Vote111111111111111111111111111111111111111");

#[program]
pub mod voting {
    use super::*;

    pub fn create_poll(
        ctx: Context<CreatePoll>,
        question: String,
        option_a: String,
        option_b: String,
    ) -> Result<()> {
        let poll = &mut ctx.accounts.poll;
        poll.question  = question;
        poll.option_a  = option_a;
        poll.option_b  = option_b;
        poll.votes_a   = 0;
        poll.votes_b   = 0;
        poll.authority = ctx.accounts.authority.key();
        poll.is_open   = true;
        Ok(())
    }

    pub fn vote(ctx: Context<CastVote>, choice: u8) -> Result<()> {
        let poll = &mut ctx.accounts.poll;
        require!(poll.is_open, PollError::PollClosed);
        match choice {
            0 => poll.votes_a += 1,
            1 => poll.votes_b += 1,
            _ => return err!(PollError::InvalidChoice),
        }
        Ok(())
    }

    pub fn close_poll(ctx: Context<ClosePoll>) -> Result<()> {
        let poll = &mut ctx.accounts.poll;
        require!(ctx.accounts.authority.key() == poll.authority, PollError::NotAuthority);
        poll.is_open = false;
        Ok(())
    }
}

#[derive(Accounts)]
pub struct CreatePoll<'info> {
    #[account(
        init,
        payer = authority,
        space = 8 + Poll::INIT_SPACE,
        seeds = [b"poll", authority.key().as_ref()],
        bump
    )]
    pub poll: Account<'info, Poll>,
    #[account(mut)]
    pub authority: Signer<'info>,
    pub system_program: Program<'info, System>,
}

#[derive(Accounts)]
pub struct CastVote<'info> {
    #[account(mut, seeds = [b"poll", authority.key().as_ref()], bump)]
    pub poll: Account<'info, Poll>,
    pub authority: Signer<'info>,
}

#[derive(Accounts)]
pub struct ClosePoll<'info> {
    #[account(mut, seeds = [b"poll", authority.key().as_ref()], bump)]
    pub poll: Account<'info, Poll>,
    pub authority: Signer<'info>,
}

#[account]
#[derive(InitSpace)]
pub struct Poll {
    pub authority: Pubkey,
    #[max_len(200)] pub question: String,
    #[max_len(100)] pub option_a: String,
    #[max_len(100)] pub option_b: String,
    pub votes_a: u64,
    pub votes_b: u64,
    pub is_open: bool,
}

#[error_code]
pub enum PollError {
    #[msg("Le sondage est ferme")] PollClosed,
    #[msg("Choix invalide : 0 ou 1 uniquement")] InvalidChoice,
    #[msg("Seul l'autorite du sondage peut le fermer")] NotAuthority,
}
"#;

println!("Exercice 2 - Programme de vote Anchor complete :");
println!("{}", VOTE_PROGRAM_COMPLET);

Exercice 2 - Programme de vote Anchor complete :


use anchor_lang::prelude::*;


declare_id!("Vote111111111111111111111111111111111111111");


#[program]


pub mod voting {


    use super::*;


... (72 outputs left out)

    #[msg("Le sondage est ferme")] PollClosed,


    #[msg("Choix invalide : 0 ou 1 uniquement")] InvalidChoice,


    #[msg("Seul l'autorite du sondage peut le fermer")] NotAuthority,


}


### Exemple guide 3 : Modeliser un Counter Solana en Rust

_Exemple contribue par le groupe **Mehdi Azouz, Ryad Gazenay & Hani Bouzida** (PR #2189)._

Pour valider la comprehension du **modèle de comptes Solana**, on modelise en Rust pur le 
programme Counter Anchor : `initialize` créé un compte `count = 0`, `increment`/`decrement` 
verifient l'authority avant toute modification, et `decrement` protege contre l'underflow 
(comme le ferait Anchor).

In [13]:
// Exercice 3 : Modeliser un Counter Solana en Rust

struct SolanaAccount {
    owner: String,
    authority: String,
    count: u64,
}

struct CounterProgram;

impl CounterProgram {
    const PROGRAM_ID: &'static str = "Counter111111111111111111111111111111111111";

    fn initialize(&self, authority: &str) -> SolanaAccount {
        SolanaAccount {
            owner: Self::PROGRAM_ID.to_string(),
            authority: authority.to_string(),
            count: 0,
        }
    }

    fn increment(&self, counter: &mut SolanaAccount, authority: &str) -> Result<(), String> {
        if authority != counter.authority {
            return Err("Autorite invalide".to_string());
        }
        counter.count += 1;
        Ok(())
    }

    fn decrement(&self, counter: &mut SolanaAccount, authority: &str) -> Result<(), String> {
        if authority != counter.authority {
            return Err("Autorite invalide".to_string());
        }
        if counter.count == 0 {
            return Err("Underflow".to_string());
        }
        counter.count -= 1;
        Ok(())
    }
}

// --- Tests ---
let program = CounterProgram;
let alice = "alice_pubkey_111111";

let mut counter = program.initialize(alice);
println!("Initialise  : count = {}", counter.count); // attendu : 0
program.increment(&mut counter, alice).unwrap();
println!("Increment   : count = {}", counter.count); // attendu : 1
program.increment(&mut counter, alice).unwrap();
println!("Increment   : count = {}", counter.count); // attendu : 2
program.decrement(&mut counter, alice).unwrap();
println!("Decrement   : count = {}", counter.count); // attendu : 1
if let Err(e) = program.increment(&mut counter, "intrus_pubkey_000") {
    println!("Autorite invalide interceptee : {}", e);
};

Initialise  : count = 0


Increment   : count = 1


Increment   : count = 2


Decrement   : count = 1


Autorite invalide interceptee : Autorite invalide


### Exercice (a completer) : Etendre le Counter avec un plafond et un reset

A votre tour ! En vous inspirant de l'exemple guide 3, faites evoluer la classe 
`CounterProgram` :

1. Ajoutez une borne `MAX_COUNT = 10` : `increment` doit lever une exception (interceptee) 
si l'on depasse le plafond.
2. Ajoutez une méthode `reset(counter, authority)` qui remet `count` a 0 -- **seule** l'authority 
du compte peut la declencher.
3. Ecrivez un petit scénario de test qui incremente jusqu'au plafond, tente un increment de trop 
(try/except), puis effectue un reset.

**Indice** : reprenez le pattern de verification `authority != counter.data["authority"]` déjà 
utilise dans `increment`/`decrement`.

In [14]:
// Exercice : etendre CounterProgram avec MAX_COUNT (plafond) et reset(counter, authority)
// TODO etudiant :
//   Etape 1 : sous-classer ou modifier CounterProgram avec MAX_COUNT = 10
//   Etape 2 : increment leve une exception (interceptee) si count atteindrait le plafond
//   Etape 3 : reset(counter, authority) remet count a 0, reserve a l'authority
//   Etape 4 : scenario de test (increment jusqu'au plafond, increment de trop, reset)

println!("Exercice a completer");

Exercice a completer


***

## 8. Resume

| Concept | Description |
|---------|-------------|
| Program | Code executable stateless |
| Account | Stockage de données |
| PDA | Adresse derivee du programme |
| CPI | Appel inter-programmes |
| Anchor | Framework Rust pour Solana |

***

**Notebook suivant** : [SC-23-Cross-Chain-Python](../06-Real-World/SC-23-Cross-Chain-Python.ipynb) | [jumeau Python](SC-22-Solana-Anchor-Python.ipynb)

***

[<< Move & Sui](SC-21-Move-Sui-Python.ipynb) | [Cross-Chain >>](../06-Real-World/SC-23-Cross-Chain-Python.ipynb) | [jumeau Python](SC-22-Solana-Anchor-Python.ipynb)

### Interpretation : Token Escrow avec CPI

Cet exercice presente un escrow atomique (echange de tokens sans tiers de confiance) utilisant des CPIs.

| Fonction | Opération | CPI utilise |
|----------|-----------|-------------|
| `make` | Initialise l'escrow et transfere Token A vers vault | `token::transfer` (maker → vault) |
| `take` | Echange atomique : Token B → maker, Token A → depositor | 2x `token::transfer` (depositor → maker, vault → depositor) |

**Points clés** :
1. `make` crée un compte escrow PDA avec les seeds `[b"escrow", maker.key(), seed]` (PDA déterministe) : le compte est créé via `init` dans la struct `Make`, et `maker` est l'auteur de la signature qui autorise ce compte
2. Le vault est un compte token PDA contrôle par l'escrow (pas de cle privee)
3. `take` utilise `CpiContext::new_with_signer` avec les seeds de l'escrow PDA pour autoriser le transfert depuis le vault
4. L'echange est atomique : soit les 2 transferts reussissent, soit aucun (revert si l'un echoue)

**Securite** :
- Le seed `u64` permet a un utilisateur d'avoir plusieurs escrows simultanes
- Le bump `ctx.bumps.escrow` est inclus dans la signature CPI pour prouver l'autorite du PDA
- L'ordre des transferts est important : d'abord le token B du depositor, puis le token A du vault (evite certains attaques)

> **Note technique** : Sur Solana, les escrows sont des PDAs (pas de contrats intelligents avec état). Le PDA signe les transferts via `new_with_signer`, ce qui est impossible sur EVM.

## Resume et perspectives

Ce notebook a permis de decourir l'ecosysteme Solana et le framework Anchor comme alternative a l'ecosysteme Ethereum/Solidity. Nous avons compare les architectures : modèle de comptes stateless et exécution BPF haute performance (~65 000 TPS théoriques) chez Solana versus stockage dans le contrat et EVM chez Ethereum. Nous avons implemente un programme Counter en Rust avec Anchor, explorant les annotations `#[program]`, `#[account]` et `#[derive(Accounts)]` qui abstraient la complexite du modèle de comptes. Nous avons etudie les PDAs (Program Derived Addresses) pour le stockage déterministe et securise de données utilisateur, les CPIs (Cross-Program Invocations) pour la composition de programmes, et implémente un escrow atomique de tokens SPL combinant `CpiContext::new` et `CpiContext::new_with_signer` pour les transfers signes par PDA.

La différence philosophique majeure entre Solana et Ethereum reside dans la separation stricte entre code (programmes stateless) et données (comptes independants). Ce modèle enable le parallelisme natif des transactions et elimine la contention globale, mais exige une planification plus rigoureuse de la structure des comptes. Les PDAs remplacent les mappings Solidity par des adresses déterministes sans cle privee, offrant une securite inherente contre les acces non autorises. Le framework Anchor simplifie considerablement le développement en Rust tout en preservant les garanties de securite du type system.

Le notebook suivant, [SC-23-Cross-Chain-Python](../06-Real-World/SC-23-Cross-Chain-Python.ipynb), aborde l'interoperabilite entre blockchains et les protocols de communication cross-chain comme Chainlink CCIP.

> Carnet issu de l'EPIC #26534.